# Word Embedding using Gensim

In [1]:
import gensim
import gensim.downloader as api

In [2]:
list(api.info()['models'].keys())

['fasttext-wiki-news-subwords-300',
 'conceptnet-numberbatch-17-06-300',
 'word2vec-ruscorpora-300',
 'word2vec-google-news-300',
 'glove-wiki-gigaword-50',
 'glove-wiki-gigaword-100',
 'glove-wiki-gigaword-200',
 'glove-wiki-gigaword-300',
 'glove-twitter-25',
 'glove-twitter-50',
 'glove-twitter-100',
 'glove-twitter-200',
 '__testing_word2vec-matrix-synopsis']

In [3]:
# This is comparatively small, but it is fast to load and good for testing
# Have around 1.2 million word vectors, each with 25 dimensions
wv = api.load("glove-twitter-25") # 25-dimensional GloVe vectors trained on Twitter data
print(wv['king']) # Get the vector for the word "king"

[==================================================] 100.0% 104.8/104.8MB downloaded
[-0.74501  -0.11992   0.37329   0.36847  -0.4472   -0.2288    0.70118
  0.82872   0.39486  -0.58347   0.41488   0.37074  -3.6906   -0.20101
  0.11472  -0.34661   0.36208   0.095679 -0.01765   0.68498  -0.049013
  0.54049  -0.21005  -0.65397   0.64556 ]


In [4]:
wv.similarity(w1 = "great", w2 = "good")

np.float32(0.9378517)

In [5]:
wv.most_similar("king", topn=5)

[('prince', 0.9337409734725952),
 ('queen', 0.9202421307563782),
 ('aka', 0.9176921844482422),
 ('lady', 0.9163240790367126),
 ('jack', 0.9147354364395142)]

In [12]:
wv.most_similar(positive=["king", "woman"], negative=["man"])

[('meets', 0.8841923475265503),
 ('prince', 0.832163393497467),
 ('queen', 0.8257461190223694),
 ('’s', 0.8174097537994385),
 ('crow', 0.813499391078949),
 ('hunter', 0.8131037950515747),
 ('father', 0.8115833401679993),
 ('soldier', 0.81113600730896),
 ('mercy', 0.8082392811775208),
 ('hero', 0.8082262277603149)]

In [ ]:
wv.most_similar(positive=["france", "berlin"], negative=["paris"])

[('frankfurt', 0.8528209924697876),
 ('hamburg', 0.8513832092285156),
 ('deutschland', 0.8492450714111328),
 ('münchen', 0.8251250982284546),
 ('düsseldorf', 0.8218944668769836),
 ('hannover', 0.8195232152938843),
 ('köln', 0.8042037487030029),
 ('bremen', 0.797916054725647),
 ('olympia', 0.7926302552223206),
 ('wien', 0.7905245423316956)]

In [13]:
wv.doesnt_match(["breakfast", "cereal", "dinner", "lunch"])

'cereal'

## Doc2Vec

Doc2Vec extends Word2Vec to generate vector representations for entire documents or paragraphs, making it useful for tasks like document classification and clustering.

In [4]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# Sample corpus
documents = [TaggedDocument(words=["hello", "world"], tags=[0]),
TaggedDocument(words=["machine", "learning"], tags=[1])]

# Train Doc2Vec model
model = Doc2Vec(documents, vector_size=50, window=2, min_count=1, workers=4)

# Infer vector for a new document
print(model.infer_vector(["hello", "machine"]))

[ 8.5007856e-03 -8.9311861e-03  6.5917075e-03 -8.0425263e-04
  4.1809333e-03 -2.9698210e-03 -4.2697303e-03 -3.6949969e-03
 -7.8552999e-03 -2.0409513e-03 -6.3263928e-03  7.0855930e-03
  6.8369210e-03  5.5759833e-03  4.2948816e-03  1.5077007e-03
  4.3677008e-03 -2.3400068e-04 -3.5799600e-03 -3.9321189e-03
  9.2592407e-03 -6.1854962e-03  7.7478886e-05  3.1868124e-03
 -1.3741339e-03 -6.6967737e-03 -2.2610975e-03 -3.8816780e-03
 -8.0909990e-03 -8.7119089e-03  5.5885874e-03 -4.6771248e-03
  9.4166221e-03  5.9195291e-03  4.9345102e-03 -2.9680347e-03
  8.6500440e-03 -1.4345491e-03  7.2299684e-03 -7.3199845e-03
  1.6077400e-04  6.5675522e-03  5.4409076e-03 -9.2571331e-03
  5.1794159e-03  9.3158484e-03 -1.9730425e-03  5.8609843e-03
  4.2925120e-04  1.4186001e-03]


# Train Doc2Vec on a corpus, then compare two paragraphs

In [5]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity

# 1) Your training corpus (replace with your paragraphs/documents)
corpus = [
    "Supply and install reinforced concrete slab 20MPa including formwork.",
    "Provide GI pipes 50mm diameter with fittings and supports.",
    "Install electrical conduits and cable trays including accessories.",
    "HVAC ducting supply and installation with insulation.",
]

# 2) Convert to TaggedDocument
train_docs = [TaggedDocument(words=simple_preprocess(text), tags=[str(i)])
              for i, text in enumerate(corpus)]

# 3) Train Doc2Vec
model = Doc2Vec(
    documents=train_docs,
    vector_size=200,
    window=8,
    min_count=1,
    workers=4,
    epochs=40,
    dm=1,          # 1 = PV-DM, 0 = PV-DBOW
    negative=10
)

# 4) Two new paragraphs to compare
p1 = "Supply and install 20 MPa RCC slab with shuttering and finishing."
p2 = "Provide 50mm galvanized iron pipe including fittings and clamps."

# 5) Infer vectors (important: inference, not training tags)
v1 = model.infer_vector(simple_preprocess(p1), epochs=50)
v2 = model.infer_vector(simple_preprocess(p2), epochs=50)

# 6) Cosine similarity
sim = cosine_similarity([v1], [v2])[0][0]
print("Similarity:", sim)

Similarity: 0.52568156
